# Gold Dataset Creation & Subdataset Cleaning Pipeline

This notebook implements an end-to-end pipeline to:
1. Load the unified arXiv dataset.
2. Define three subdataset selection strategies (Top N, Water-filling, Quartile).
3. Create Gold Datasets: For each strategy, iteratively select articles, download their LaTeX source, and extract text chunks containing citations until 20 valid articles are processed.
4. Clean Subdatasets: Generate the final subdatasets with the 20 Gold articles excluded.
5. Save all outputs in JSON format.


In [ ]:
import json
import os
import re
import random
import tarfile
import gzip
import shutil
import time
import requests
import io
from typing import List, Dict, Set, Any, Tuple, Optional
from collections import defaultdict
from difflib import SequenceMatcher

: 

## 1. Configuration & Helper Functions

We define paths and regex patterns for parsing LaTeX citations. The `clean_text` function removes LaTeX formatting to leave readable text.

In [ ]:
DATA_PATH = 'data/processed/unified_articles.json'
OUTPUT_DIR = 'data/gold_datasets_linked/'
TEMP_DIR = 'temp_latex_downloads_linked/'
TARGET_GOLD_SIZE = 20
CONTEXT_WINDOW = 400

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
if not os.path.exists(TEMP_DIR):
    os.makedirs(TEMP_DIR)

# --- Pre-compiled Regex ---
# Matches LaTeX citations like \cite{foo}, \citep{foo, bar}
CITE_PATTERN = re.compile(r'(\\(cite|citep|citet|bibcite)\{([^}]+)\})')

# Matches bibliography items: \bibitem{key} The Title ...
BIBITEM_PATTERN = re.compile(r'\\bibitem\{([^}]+)\}([\s\S]*?)(?=\\bibitem|\\end\{thebibliography\})')

def clean_latex_text(text: str) -> str:
    """Cleans LaTeX formatting for readability."""
    if not text: return ""
    # Remove comments
    text = re.sub(r'%.*', '', text)
    # Simplify common commands
    text = re.sub(r'\\(textbf|textit|emph|section|subsection)\{([^}]+)\}', r'\2', text)
    text = re.sub(r'\$([^$]+)\$', r'\1', text) # Inline math
    # Normalize whitespace
    return re.sub(r'\s+', ' ', text).strip()

## 2. Data Loading

Load the unified dataset containing article metadata and references.

In [ ]:
def load_and_index_data(filepath: str) -> Tuple[List[Dict], Dict[str, str]]:
    """
    Loads data and creates a global mapping of ID -> Title.
    This allows us to look up the titles of referenced IDs.
    """
    print(f"Loading dataset from {filepath}...")
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    articles = data['articles'] if isinstance(data, dict) and 'articles' in data else data
    
    print("Building ID -> Title index...")
    id_to_title = {}
    for art in articles:
        if 'id' in art and 'title' in art:
            # Normalize title for better matching (lowercase, no punctuation)
            clean_t = re.sub(r'[^a-z0-9\s]', '', art['title'].lower())
            id_to_title[art['id']] = clean_t
            
    print(f"Indexed {len(id_to_title)} titles.")
    return articles, id_to_title

articles_data, global_id_to_title = load_and_index_data(DATA_PATH)

## 3. Article Download & Processing

These functions handle the complexity of retrieving source files from arXiv:
1.  `download_source`: Fetches the `.tar.gz` or `.pdf` from arXiv.
2.  `extract_tex_content`: Unpacks the source and finds `.tex` files.
3.  `extract_citation_chunks`: Scans the `.tex` content for citations and extracts the surrounding context.

In [ ]:
def download_source(arxiv_id: str, save_dir: str) -> Optional[str]:
    """Downloads .tar.gz source from arXiv."""
    url = f"https://arxiv.org/e-print/{arxiv_id}"
    save_path = os.path.join(save_dir, f"{arxiv_id}.tar.gz")
    try:
        response = requests.get(url, stream=True, timeout=20)
        if response.status_code == 200 and 'pdf' not in response.headers.get('content-type', ''):
            with open(save_path, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            return save_path
    except Exception as e:
        print(f"Error downloading {arxiv_id}: {e}")
    return None

def extract_source_content(file_path: str) -> Tuple[str, str]:
    """
    Extracts:
    1. Full combined LaTeX body (for text extraction)
    2. Combined Bibliography content (from .bbl files or embedded \thebibliography)
    """
    full_body = ""
    full_bib = ""
    
    try:
        if tarfile.is_tarfile(file_path):
            with tarfile.open(file_path, 'r:*') as tar:
                for member in tar.getmembers():
                    # Extract Text
                    if member.name.endswith('.tex'):
                        try:
                            f = tar.extractfile(member)
                            content = f.read().decode('utf-8', errors='ignore')
                            full_body += f"\n% --- FILE: {member.name} ---\n{content}"
                            # Check for embedded bibliography
                            if '\\begin{thebibliography}' in content:
                                full_bib += content
                        except: pass
                    
                    # Extract Bibliography file
                    elif member.name.endswith('.bbl'):
                        try:
                            f = tar.extractfile(member)
                            full_bib += f.read().decode('utf-8', errors='ignore') + "\n"
                        except: pass
        else:
            # Single file case
            with gzip.open(file_path, 'rt', encoding='utf-8', errors='ignore') as f:
                content = f.read()
                full_body = content
                full_bib = content
                
    except Exception as e:
        print(f"Error extracting {file_path}: {e}")
        
    return full_body, full_bib

## 4. Citation Resolution Logic

This is the core intelligence. We map `\cite{key}` -> `Bib Title` -> `ArXiv ID`.

1.  **Parse Bibliography**: Extract `(key, raw_text_title)` pairs from LaTeX.
2.  **Fuzzy Match**: Compare the extracted title against the titles of the IDs listed in the article's `refs`.

In [ ]:
def parse_bibliography_keys(bib_content: str) -> Dict[str, str]:
    """
    Parses \bibitem{key} ... entries.
    Returns map: { cite_key: raw_bib_text }
    """
    bib_map = {}
    for match in BIBITEM_PATTERN.finditer(bib_content):
        key = match.group(1).strip()
        text = match.group(2)
        # Clean up the text to find a title
        # We assume the title is the longest segment or use a simple heuristic
        clean_text = re.sub(r'\\(newblock|emph|textit|textbf)', ' ', text)
        clean_text = re.sub(r'[^a-zA-Z0-9\s]', '', clean_text.lower())
        bib_map[key] = clean_text
    return bib_map

def resolve_citations(bib_map: Dict[str, str], known_ref_ids: List[str]) -> Dict[str, str]:
    """
    Links citation keys to real ArXiv IDs using fuzzy title matching.
    
    Args:
        bib_map: {cite_key: bib_text_from_latex}
        known_ref_ids: List of ArXiv IDs that this paper actually cites (from dataset)
    
    Returns:
        key_to_id: {cite_key: arxiv_id}
    """
    key_to_id = {}
    
    # 1. Retrieve titles for known reference IDs
    candidate_titles = {}
    for rid in known_ref_ids:
        if rid in global_id_to_title:
            candidate_titles[rid] = global_id_to_title[rid]

    if not candidate_titles:
        return {}

    # 2. Match keys to IDs
    for key, bib_text in bib_map.items():
        best_score = 0.0
        best_id = None
        
        # Optimization: Bib text is usually long. Title is a substring.
        # We check if the candidate title exists roughly inside the bib text.
        for rid, title in candidate_titles.items():
            # Fast check: is title subset of bib text?
            if title in bib_text:
                score = 1.0
            else:
                # Slow check: SequenceMatcher
                # We only match first 200 chars of bib entry to save time
                score = SequenceMatcher(None, title, bib_text[:300]).ratio()
            
            if score > best_score:
                best_score = score
                best_id = rid
        
        # Threshold for a valid match
        if best_score > 0.65: 
            key_to_id[key] = best_id
            
    return key_to_id

## 4. Strategy Implementations

We implement the three required subdataset creation strategies to generate candidate lists.

1.  **Most Cited (Top N)**: Simple sort by citation count.
2.  **Stratified (Water Filling)**: Balanced selection across categories.
3.  **Quartile**: Selection based on citation quartiles. Each quartile takes 25%.

In [ ]:
def get_candidates_most_cited(articles: List[Dict], top_n: int = 50000) -> List[Dict]:
    """Returns top N articles sorted by number of references (or citations if available)."""
    # Sorting by number of OUTGOING references as a proxy if citation count isn't in metadata.
    # If you have an 'citations_count' field, use that instead.
    # Assuming 'refs' field exists from preprocessing.
    sorted_articles = sorted(articles, key=lambda x: len(x.get('refs', [])), reverse=True)
    return sorted_articles[:top_n]

def get_candidates_stratified(articles: List[Dict], total_k: int = 50000) -> List[Dict]:
    """Returns K articles distributed across categories (Water Filling)."""
    # 1. Group by category
    cat_map = defaultdict(list)
    for art in articles:
        cats = art.get('categories', '').split()
        if cats:
            primary_cat = cats[0]
            cat_map[primary_cat].append(art)
            
    # 2. Calculate target per category
    n_cats = len(cat_map)
    if n_cats == 0: return []
    target_per_cat = max(1, total_k // n_cats)
    
    selected = []
    # 3. Select top cited from each category
    for cat, arts in cat_map.items():
        # Sort by refs count
        sorted_arts = sorted(arts, key=lambda x: len(x.get('refs', [])), reverse=True)
        selected.extend(sorted_arts[:target_per_cat])
        
    return selected[:total_k]

def get_candidates_balanced_quartiles(articles: List[Dict], total_n: int = 50000) -> List[Dict]:
    """
    Creates a subdataset of size `total_n` composed of equal shares from each 
    citation quartile of the original dataset.
    
    - 25% of `total_n` from Q1 (Top 25% most cited)
    - 25% of `total_n` from Q2
    - 25% of `total_n` from Q3
    - 25% of `total_n` from Q4 (Bottom 25% least cited)
    """
    # 1. Sort all articles by citation count (Descending: Most cited -> Least cited)
    #    Using 'refs' length as proxy for citations if 'citations_count' is unavailable
    sorted_articles = sorted(articles, key=lambda x: len(x.get('refs', [])), reverse=True)
    
    total_source = len(sorted_articles)
    if total_source == 0:
        return []

    # 2. Define the size of one quartile in the SOURCE dataset
    source_q_size = total_source // 4
    
    # 3. Define how many items we want from each quartile in the OUTPUT
    target_per_quartile = total_n // 4
    
    balanced_selection = []
    
    # 4. Iterate through the 4 quartiles
    for q in range(4):
        # Define start/end indices for this quartile in the sorted source list
        start_idx = q * source_q_size
        
        # For the last quartile, ensure we go to the very end (handle remainder)
        if q == 3:
            end_idx = total_source
        else:
            end_idx = start_idx + source_q_size
            
        # Extract the full pool for this quartile
        quartile_pool = sorted_articles[start_idx:end_idx]
        
        # Select the top portion of this specific quartile to meet our target
        # (Since pool is sorted, this picks the 'best' of the quartile. 
        #  Use random.sample(quartile_pool, target_per_quartile) if you want random sampling within the quartile)
        selection = quartile_pool[:target_per_quartile]
        
        balanced_selection.extend(selection)
        
    # 5. Handle Rounding Errors (if total_n isn't divisible by 4)
    #    Fill any remaining slots with the highest cited papers not yet selected (from Q1)
    missing = total_n - len(balanced_selection)
    if missing > 0:
        # We took the first 'target_per_quartile' from Q1. 
        # The next best candidates start immediately after that index.
        remainder_start = target_per_quartile
        remainder_end = remainder_start + missing
        
        # Ensure we don't go out of bounds of Q1
        if remainder_end < source_q_size:
            balanced_selection.extend(sorted_articles[remainder_start:remainder_end])
    
    return balanced_selection

## 5. Main Processing Pipeline

The `create_gold_dataset` function is the engine. It:
1. Takes a candidate list.
2. Iterates through candidates, trying to download and parse them.
3. Stops when 20 valid articles (with citations found) are collected.
4. Returns the Gold List and the IDs to be removed from the subdataset.

In [ ]:
def process_article_for_gold(article: Dict) -> Optional[Dict]:
    """
    Full pipeline for a single article.
    Returns dict {id: ..., chunks: {ref_id: [text, text]}} if successful.
    """
    aid = article['id']
    known_refs = article.get('refs', [])
    if not known_refs:
        return None

    # 1. Download
    source_path = download_source(aid, TEMP_DIR)
    if not source_path: return None
    
    # 2. Extract
    tex_body, bib_content = extract_source_content(source_path)
    
    # 3. Parse & Resolve Keys
    bib_keys_map = parse_bibliography_keys(bib_content)
    key_to_arxiv_id = resolve_citations(bib_keys_map, known_refs)
    
    if not key_to_arxiv_id:
        # Cleanup
        if os.path.exists(source_path): os.remove(source_path)
        return None
        
    # 4. Extract Chunks linked to IDs
    # Structure: { ref_id: [chunk1, chunk2] }
    chunks_by_id = defaultdict(list)
    
    for match in CITE_PATTERN.finditer(tex_body):
        cite_keys_str = match.group(3)
        # Handle multiple keys: \cite{key1, key2}
        keys = [k.strip() for k in cite_keys_str.split(',')]
        
        # Get context
        start, end = match.span()
        context_start = max(0, start - CONTEXT_WINDOW)
        context_end = min(len(tex_body), end + CONTEXT_WINDOW)
        raw_text = tex_body[context_start:context_end]
        clean_text = clean_latex_text(raw_text)
        
        # Assign this text to every valid ID found in the keys
        for k in keys:
            if k in key_to_arxiv_id:
                target_id = key_to_arxiv_id[k]
                chunks_by_id[target_id].append(clean_text)
                
    # Cleanup
    if os.path.exists(source_path): os.remove(source_path)
    
    if chunks_by_id:
        return {
            "id": aid,
            "citations": dict(chunks_by_id) # Convert defaultdict to dict for JSON
        }
    return None

def run_gold_creation_pipeline(strategies:Dict, articles_data: List[Dict]):
    # Strategies from previous step (re-defined here for completeness)
    for strat_name, strat_func in strategies.items():
        print(f"\n=== Running strategy: {strat_name} ===")
        candidates = strat_func(articles_data)
        # Saving subdataset
        json.dump(candidates, open(os.path.join(OUTPUT_DIR, f'subdataset_{strat_name}.json'), 'w'), indent=2)
        print(f"Saving {len(candidates)} candidate articles to subdataset_{strat_name}.json")
        gold_dataset = []
        gold_ids_to_remove = []
        
        print(f"Processing candidates to find {TARGET_GOLD_SIZE} valid Gold articles...")
        
        for art in candidates:
            if len(gold_dataset) >= TARGET_GOLD_SIZE:
                break
                
            result = process_article_for_gold(art)
            if result:
                gold_dataset.append(result)
                gold_ids_to_remove.append(art['id'])
                print(f"Added {art['id']} ({len(result['citations'])} resolved refs)")
            else:
                print(f"Skipped {art['id']} (No parsed refs)")
                
        # Save
        with open(os.path.join(OUTPUT_DIR, f'gold_dataset_{strat_name}.json'), 'w') as f:
            json.dump(gold_dataset, f, indent=2)

        print("Done.")


In [ ]:
# ==========================================
# EXECUTION PHASE
# ==========================================

strategies = {
    "quartile_q1": lambda data: get_candidates_balanced_quartiles(data, total_n=50000),
    # "most_cited": lambda data: get_candidates_most_cited(data, top_n=50000),
    # "stratified": lambda data: get_candidates_stratified(data, total_k=50000)
}

run_gold_creation_pipeline(strategies, articles_data)

In [ ]:
# Erasing articles used in gold from each subdataset:
for strat_name in strategies.keys():
    print(f"\n=== Cleaning subdataset for strategy: {strat_name} ===")
    # Load gold dataset
    gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_{strat_name}.json')
    if not os.path.exists(gold_path):
        print(f"Gold dataset {gold_path} not found, skipping.")
        continue
    with open(gold_path, 'r') as f:
        gold_data = json.load(f)
    gold_ids = set([art['id'] for art in gold_data])
    
    # Load subdataset
    subdataset_path = os.path.join(OUTPUT_DIR, f'subdataset_{strat_name}.json')
    with open(subdataset_path, 'r') as f:
        subdataset_data = json.load(f)
    
    # Filter out gold articles
    clean_subdataset = [art for art in subdataset_data if art['id'] not in gold_ids]
    
    # Save cleaned subdataset
    clean_path = os.path.join(OUTPUT_DIR, f'clean_subdataset_{strat_name}.json')
    with open(clean_path, 'w') as f:
        json.dump(clean_subdataset, f, indent=2)
    
    print(f"Cleaned subdataset saved to {clean_path}, size reduced from {len(subdataset_data)} to {len(clean_subdataset)} articles.")

In [ ]:

# In each gold dataset, we remove any string like \cite{...} to avoid leakage.
for strat_name in strategies.keys():
    print(f"\n=== Cleaning citations in gold dataset for strategy: {strat_name} ===")
    gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_{strat_name}.json')
    if not os.path.exists(gold_path):
        print(f"Gold dataset {gold_path} not found, skipping.")
        continue
    with open(gold_path, 'r') as f:
        gold_data = json.load(f)
    
    # Clean citations in chunks
    for art in gold_data:
        for ref_id, chunks in art['citations'].items():
            cleaned_chunks = []
            for chunk in chunks:
                cleaned_chunk = CITE_PATTERN.sub('', chunk)
                cleaned_chunks.append(cleaned_chunk)
            art['citations'][ref_id] = cleaned_chunks
    
    # Save cleaned gold dataset
    cleaned_gold_path = os.path.join(OUTPUT_DIR, f'gold_dataset_cleaned_{strat_name}.json')
    with open(cleaned_gold_path, 'w') as f:
        json.dump(gold_data, f, indent=2)
    
    print(f"Cleaned gold dataset saved to {cleaned_gold_path}.")

In [ ]:
# Cleanup Temp Directory
shutil.rmtree(TEMP_DIR, ignore_errors=True)
print("\nAll tasks completed. Temporary files removed.")